<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 5 - Features importances</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [1]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [2]:

# Roots
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
import os
from dotenv import load_dotenv
import gc

# Selection
from sklearn.model_selection import train_test_split

# feature importance
import shap

In [3]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [4]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))

In [5]:
# Fonctions personnelles

# utils
from notebooks.utils.features_type_list import features_type
from notebooks.utils.get_feature_names_out import get_feat_names
from notebooks.plotting.config_figures import save_figure


In [6]:

# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

# variables globales
random_state=42

# test size
test_size = 1000

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traquer les expérimentations avec MLFlow </span>

In [7]:
# import
import mlflow

# ================== Ce qui est dit sur la docs de MLFlow ==============
# 1. Config Tracking
# database en local (sqlite)
# mlflow.set_tracking_uri("sqlite:///mlflow.db")
# mlflow.set_experiment("my-first-experiment")
# database distant
# # Connect to remote MLflow server
# mlflow.set_tracking_uri("http://localhost:5000")
# mlflow.set_experiment("my-first-experiment")
# # ou
# export MLFLOW_TRACKING_URI="http://localhost:5000"
# export MLFLOW_EXPERIMENT_NAME="my-first-experiment"
# =====================================================================

mlflow.set_tracking_uri("sqlite:///mlflow.db")
# On définie l'experiment
experiment_name = "Feature_importance"
mlflow.set_experiment(experiment_name)

2026/01/30 19:48:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/30 19:48:15 INFO mlflow.store.db.utils: Updating database tables
2026/01/30 19:48:15 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/30 19:48:15 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/30 19:48:15 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/30 19:48:15 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/30 19:48:16 INFO mlflow.tracking.fluent: Experiment with name 'Feature_importance' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/notebooks/mlruns/3', creation_time=1769798896037, experiment_id='3', last_update_time=1769798896037, lifecycle_stage='active', name='Feature_importance', tags={}>

In [8]:
# 2. Vérif connexion

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: {mlflow.get_experiment_by_name('Feature_importance')}")

# Test logging
with mlflow.start_run():
    mlflow.log_param("test_param", "test_value")
    print("✓ Successfully connected to MLflow!")

MLflow Tracking URI: sqlite:///mlflow.db
Active Experiment: <Experiment: artifact_location='/home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/notebooks/mlruns/3', creation_time=1769798896037, experiment_id='3', last_update_time=1769798896037, lifecycle_stage='active', name='Feature_importance', tags={}>
✓ Successfully connected to MLflow!


**A lancer sur le terminal**
```python
# Accès MLFlow UI

# For Option A (local database)
mlflow server \
    --backend-store-uri sqlite:///mlflow.db \
    --default-artifact-root ./mlruns \
    --host 127.0.0.1 \
    --port 5000
# # For Option B (distant database)
# If you have the remote tracking server running (option C), access the MLflow UI at the same URI.
```

<span style="color:purple"> IMPORTANT - pour moi-même: Suivant la méthodologie employée (utilisation de Kaggle pour la simulation), les résultats de MLFlow sont dans le dossier "**export_mlflow_complet**". Cette précision est nécéssaire car du fait des particlarités de Kaggle (exemple des paths), simplement rappatrier les résultats crééent un conflit au niveau des chemins des fichiers. Pour éviter cela, on a comapcter dans le dossier en question est on utilise la commande:
- **mlflow ui --backend-store-uri sqlite:///mlflow.db**

**On verra ainsi s'afficher les runs, métriques et les sauvegardes autolog. CEPENDANT, ne sera aps présent (présent dans le dossier mais pas UI) les courbes et artefacts**. Cel est dû au fait que mlflow enregistre en dur sur la db (en présence d'une db, il ne crée pas de fichier meta.yaml) d'une façon qui lors de l'export rend difficile la modification afin de changer les chemins de lecture. Ils restent cependant présent en local et en cas de simulation "normal", ils apparaissent bien dans l'UI.


In [37]:
# Enable autologging for scikit-learn
# Va save le modele, les metriques, les hyperparam et des métadonnées (temps, format...)
mlflow.sklearn.autolog() # type: ignore

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [10]:
# Chemin du dataset d'entrainement/test du modèle
datas_path = (
    root_path /'datas'/'raw_datas'/
    'Projet+Mise+en+prod+-+home-credit-default-risk'/'final_datasets'
)

<span style="color:blue;font-weight:bold">Chargemet des artefacts</span>

<span style="color:red;font-weight:bold"> Parce qu'on a dû simuler sur Kaggle puis rappatrié la donnée dans un dossier spécifique... on ne suit pas le fonctionnement "classique". d'où l'utilisation du datas_path en dur, sinon commenter cette partie et décommenter la partie de chargement suivant l'URI mlflow</span>

In [25]:
load_dotenv() # Charge les variables du .env

# Variable d'environnement
base_path_str = os.getenv('MLFLOW_EXPORT_PATH')
experiment_folder_id_str = os.getenv('EXPERIMENT_ID')
run_folder_id_str = os.getenv('RUN_ID')
model_name_str = os.getenv('MODEL_NAME') # avec le format

base_path = Path(str(base_path_str))
experiment_folder_id = Path(str(experiment_folder_id_str))
run_folder_id=Path(str(run_folder_id_str))
model_name=Path(str(model_name_str))

# Chemin standard MLflow pour les modèles
# model_path = \
    # base_path / "mlruns"/ experiment_folder_id/ run_folder_id/ "artifacts"/"model"/model_name
model_path = base_path / "mlruns"/ experiment_folder_id/ run_folder_id/ "artifacts"/model_name

# Chargement du modele
if os.path.exists(model_path):
    best_model_pipe = joblib.load(model_path)
    print(f"Modèle chargé")


Modèle chargé


In [12]:
# Importation de la donnée
Xy= pd.read_parquet(datas_path/"train.parquet")

In [13]:
# Définition des prédicteurs X et de la cible y
X = Xy.drop(columns=['TARGET'])
y = Xy['TARGET']

del Xy
gc.collect()

8

In [14]:
# Échantillonnage stratifié (1000 lignes) ==> pas nécéssaire de travailler sur l'ensemble du jeu
# pour avoir les tendances MAIS pensé a stratifié
_, X_test_shap, _, y_test_shap = train_test_split(
    X, y, test_size=test_size, stratify=y, random_state=random_state
)

del X,y
gc.collect()

print("Données bruts supprimé et échantillons prêts à l'emploi")

Données bruts supprimé et échantillons prêts à l'emploi


In [15]:
# # Identification des features numériques et catégorielles
# num_list, cat_list = features_type(X_test_shap)

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Configurations </span>

<span style="color:blue;font-weight:bold">Préprocesseur et modèle</span>

Remarque: On pourrait en théorie fournir directement a shap "best_model", mais ça signifierait qu'il ingurgite toute la pipeline ==> hausse du coût + étapes en interne inutile. On sépare donc les steps de la pipeline.

In [26]:
# Identification des steps de preproc et du classificateur
# les étapes s'appellent preprocessor et classifier, à changer si c'est autre chose
preprocessor = best_model_pipe.named_steps['preprocessor']
classifier = best_model_pipe.named_steps['classifier']

In [27]:
# On transforme les données brutes en chiffres (encodage, scaling, imputation)
X_test_shap_transformed = preprocessor.transform(X_test_shap)

In [29]:
# On récupère le nom "original" des features stockées
feature_names = get_feat_names(preprocessor)

Récupération manuelle des noms des features...


In [30]:
# Mise en dataframe
X_test_shap_df = pd.DataFrame(X_test_shap_transformed, columns=feature_names)
display(X_test_shap_df)

,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,WALLSMATERIAL_MODE_None,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_Yes,EMERGENCYSTATE_MODE_None
0,268098.0,1.0,157500.0,270000.0,13846.5,270000.0,0.018209,-10685.0,-676.0,-3152.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,164257.0,1.0,126000.0,337500.0,16875.0,337500.0,0.020713,-16043.0,-1566.0,-919.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,390778.0,1.0,135000.0,269550.0,21294.0,225000.0,0.008575,-8557.0,-383.0,-8546.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,444763.0,0.0,202500.0,886500.0,26050.5,886500.0,0.018209,-19842.0,-3150.0,-251.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,426547.0,0.0,103500.0,359685.0,28548.0,310500.0,0.010500,-9311.0,-1362.0,-4079.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,121981.0,0.0,405000.0,830443.5,78979.5,769500.0,0.072508,-17100.0,-761.0,-7395.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
996,292990.0,0.0,157500.0,225000.0,11250.0,225000.0,0.007020,-20065.0,-1531.0,-4747.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
997,240758.0,1.0,180000.0,268659.0,17950.5,243000.0,0.010276,-12863.0,-1890.0,-6924.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
998,381162.0,0.0,90000.0,276277.5,14233.5,238500.0,0.007120,-14646.0,-2291.0,-1556.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


<span style="color:blue;font-weight:bold">Explainer et Shapleys</span>

In [31]:
# Vérifier que l'explainer est adapté au modèle
explainer = shap.TreeExplainer(classifier)
shap_values = explainer(X_test_shap_df)

In [32]:
# IMPORTANT : On réinjecte les données brutes dans l'objet SHAP 
# pour que les graphes affichent les vraies valeurs (ex: "Age = 45" et pas "Age = 0.87 normalized")
shap_values.data = X_test_shap_df.values
shap_values.feature_names = list(feature_names)

del X_test_shap_df
gc.collect()

597

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Visualisation et interprétation </span>

<span style="color:blue;font-weight:bold"> Pré-config </span>

In [33]:
# Pour le LOCAL afin d'avoir un echantillon de chaque type, on recalcul les probas et la pred
# Seuil optimisé
opt_threshold = 0.511
y_pred_proba = best_model_pipe.predict_proba(X_test_shap)[:,1]
y_pred = (y_pred_proba >= opt_threshold).astype(int)

idx_tp = np.where((y_test_shap == 1) & (y_pred == 1))[0][0] # Vrai positif
idx_tn = np.where((y_test_shap == 0) & (y_pred == 0))[0][0] # Vrai négatif 
idx_fp = np.where((y_test_shap == 0) & (y_pred == 1))[0][0] # Faux positif 
idx_fn = np.where((y_test_shap == 1) & (y_pred == 0))[0][0] # Faux négatif 

confusion_idx_dict = {
    'idx_tp':idx_tp,
    'idx_tn':idx_tn,
    'idx_fp':idx_fp,
    'idx_fn':idx_fn,
}

del opt_threshold, y_pred_proba, y_pred, idx_fn, idx_fp, idx_tn, idx_tp
gc.collect()

32

In [ ]:
# pour le SCATTER

# On récupère les indices des features triées par importance
# shap_values est l'objet Explanation
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top_indices = np.argsort(mean_abs_shap)[::-1] # Tri décroissant
top_features = [feature_names[i] for i in top_indices[:5]] # Top 5

del mean_abs_shap, top_indices
gc.collect()

0

<span style="color:blue;font-weight:bold"> Visualisation </span>

In [38]:
mlflow.autolog(disable=True) # On va controler le flux de sauvegarde manuellement

In [ ]:
# IMPORTANT A CAUSE DU FONCTIONNEMENT DE SHAP NE JAMAIS METTRE DE PLT.SHOW()
with mlflow.start_run(run_name="Analyse_comportement_features") as run:

    # ============= Feature importance_globale ======================
    # Comportement des clients (société)
    
    fig_glob = plt.figure(figsize=(12, 8))
    shap.plots.beeswarm(shap_values, max_display=10, show=False)
    plt.title("Global: Comportement des clients", fontsize=12)
    # logging
    mlflow.log_figure(
        plt.gcf(), 
        "Feature_importance_globale.png",
        save_kwargs={'bbox_inches': 'tight'}
        )
    # Sauvegarde locale
    save_figure("Feature_importance_globale", save_path/"figures")
    plt.close(fig_glob) # On ferme pour libérer la RAM
    
    # ============= Feature importance_locale ======================
    # Comportement par client
    
    for key,value in confusion_idx_dict.items():
        fig_local = plt.figure(figsize=(12,8))
        shap.plots.waterfall(
            shap_values[value],
            max_display = 10,
            show=False
        )
        plt.title("Local: Comportement par client", fontsize=12)
        # logging
        mlflow.log_figure(
            plt.gcf(), 
            f"Feature_importance_locale_cas_{key}.png",
            save_kwargs={'bbox_inches': 'tight'}
        )
        # Sauvegarde locale
        save_figure(f"Feature_importance_locale_cas_{key}", save_path/"figures")
        plt.close(fig_local)
        
    # ============= Feature importance_unit ======================
    # Evolution d'une feature et son impact sur le comportement
    
    for feature in top_features:
        fig_scatter = plt.figure(figsize=(8,6))
        shap.plots.scatter(shap_values[:, feature], show=False, color=shap_values[:, feature])
        plt.title("Unitaire: Impact des features", fontsize=12)
        # logging
        mlflow.log_figure(
            plt.gcf(), 
            f"Influence_par_feature_cas_{feature}.png",
            save_kwargs={'bbox_inches': 'tight'}
        )
        # Sauvegarde locale
        save_figure(f"Influence_par_feature_cas_{feature}", save_path/"figures")
        plt.close(fig_scatter)

Feature_importance_globale sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Feature_importance_locale_cas_idx_tp sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Feature_importance_locale_cas_idx_tn sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Feature_importance_locale_cas_idx_fp sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Feature_importance_locale_cas_idx_fn sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Influence_par_feature_cas_EXT_SOURCE_COUNT sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Influence_par_feature_cas_DAYS_EMPLOYED sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6/datas/results/figures
Influence_par_feature_cas_DAYS_BIRTH sauvegardé dans /home/shipoz/Documents/OPENCLASSROOMS/P6/livrable_P6

<span style="color:blue;font-weight:bold"> Interpretations </span>

0. **Rappel lecture d'une figure shap (en général)**:

- Les features sont classé par ordre d'importance en terme d'impact absolu sur la prédiction
- Suivant l'axe des abscisses, on regarde les shapleys qui représente, l'impact de la feature sur la décision du modèle:
    - plus on est a droite plus la feature induit dans la direction de la classe cible
    - plus on est a gauche, plus la feature tend vers la classe opposée
- Les couleurs sont rattachées à la colorbar a droite et représente les valeurs de feature
- plus on a rassemblement d'observations avec une même valeur shap, plus elle s'étend verticalement.


1. **Feature globale**

La feature importance global donne une image du comportement des clients en général.
- la feature la plus importante est "EXT_SOURCE_COUNT", la feature qui définit le nombre de source externe ayant fourni un score normalisé. Son influence est très vaste (shapleys $\in [-1.5;1.5]$). Plus la valeur de cette feature est elevée et plus la personne est considéré comme solvable. On remarque aussi que la colorimétrie est graduelle (plus on fourni de source plus la feature oriente vers une solvabilité).

- Les autres features sont beaucoup moins étalées et concentrés dans l'intervalle [-0.5;0.5] environ, on a:
    - La durée d'employabilité. Plus la durée est faible plus la feature prédit une non solvabilité avec une concentration vers 0.25 shapley (**Pour rappel, les durées sont comptées négativement donc plus la valeur est élevée, moins la durée est grande!**)
    - L'âge (date de naissance). Plus la personne est jeune plus il y a un risque.
    - Le montant des annuités. Plus elle est elevée, plus il est risqué de prêter
    - La vitesse de remboursement. La feature n'est pas très clair car on a un mix de couleur. On observe tout de même que pour lorsque la vitesse est elevée, la prédiction de solvabilité est améliorée.
    - La date la plus proche de la dernière echeance de la première version du contrat. Plus cette date est proche (rouge) plus il est risqué.
    - La moyenne de la dette totale actuelle, plus elle est faible plus il est solvable.
    - l'age de la voiture
    - être marié rend plus solvable.
- Les features, passé le statut marital marié, il y a des features dont la shap value varie dans les extrêmes et même plus que pour les sources cependant on est déjà sur des stades d'influence minimes.

2. **Feature locale**

La feature importance locale permet d'étudier au cas par cas, les clients et voir comment le modèle a jugé l'importance des ses comportements.
- EXT_SOURCE_COUNT suffit souvent à définir le ton de la prédiction ce qui induit aussi aux erreurs (dans le cas des deux faux, la feature domine totalement le reste des features).
- Les 831 features restantes ont une influence minime (au plus du même ordre que la seconde/troisième feature).

3. **Scatter**

- EXT_SOURCE_COUNT est quasi une droite affine décroissante autant en terme de shap values que de feature values. la distribution semble normale (peut etre platikurtic eventuellement)
- DAYS_BIRTH a une forme en cloche avec un pic autour des 12500 jours soit environ 34 ans. Après 17500 (47 ans), la feature oriente vers la solvabilité (**même les 25000 jours soit environ 68 ans, ce qui est improbable, sauf si ce sont tous des B.Arnault**). La distribution est plutôt platykurtic
- DAYS_EMPLOYED montre qu'en dessous de 2000 jours travaillés (5 ans), on est plutôt prédit insolvable puis jusqu'à 6000 jours (16 ans), on est de plus en plus solvable et il semble (à part deux clients) que l'impact ne progresse plus à partir de là. En terme de distribution, on est très left skewed (concentration des personnes avec peu d'années de travail)

**Conclusion**:

Au final ce qu'on relève c'est qu'il est très probable qu'on puisse se limiter a un nombre très restreint de feature (si on veut faire de l'instantanné, on pourrait juste regardé le nombre de source externe pour un premier tri, puis ajouter les 4 features suivantes si on veut affiner etc...mais passé les 10-50 premières features le reste va surtout complexifier le modèle et augmenter le coût d'entrainement). Pour se décider du nombre, on peut aller tester sur le jeu de test de Kaggle.